# Notebook 10 — Context Engineering and Inference Optimisation

**Status:** Outline / stub. Full content to be developed when course moves to its own repo.

**Prerequisites:** NB02 (llm_engines), NB04 (Engram memory), NB05/06 (RAG)

**Toolkit:** `llm_engines`, `rag_lib`, `engram`

---

## Why this notebook exists

After NB02–06 students know how to build an LLM application. This notebook
addresses the next question: **how do you make it fast and affordable, especially
on local hardware or when paying cloud API costs?**

The central thesis: the context window is a *managed resource*, not a passive
buffer. Two strategies compose to handle it:

- **Minimise what enters the context** — keep large data external; let the
  model query it selectively (RAG, RLM pattern).
- **Optimise what is in the context** — KV-cache reuse across turns and
  agent sub-calls, token budget discipline, stable prefix ordering.

---

## Section 1 — The KV cache: what it is and why it matters locally

- Brief: what attention is, what the KV cache stores, why recomputing it is
  expensive (VRAM bandwidth on a 3090 vs a cloud A100).
- The prefix-cache intuition: if consecutive calls share a long common prefix
  (system prompt, background docs), the backend only computes it once.
- Local impact: latency. Cloud impact: cost (Anthropic ~90 % reduction;
  OpenAI ~50 % reduction on cached tokens).

## Section 2 — `session_id` and `CacheStats` in `llm_engines`

**Exercise 2a:** Single-turn baseline — measure time-to-first-token and
`cache_stats.hit_ratio` (will be 0 or low without session_id).

**Exercise 2b:** Add `session_id` across turns in a short conversation.
Observe `hit_ratio` rising from turn 2 onward on a compatible backend.

**Exercise 2c:** Deliberately break the cache by inserting dynamic content
before the stable prefix. Observe `hit_ratio` returning to 0. Fix it by
restoring the correct ordering.

**The ordering rule:**
```
[system prompt] → [stable background] → [dynamic memory] → [dynamic RAG] → [user message]
```

## Section 3 — Engram token budget as a cost lever

- Walk through `ProjectMemory(token_budget=...)` and show how growing Engram
  context pushes the cached prefix boundary outward.
- Exercise: observe `hit_ratio` drop as a 10-turn session fills the Engram
  context. Tighten `TokenBudget`, observe `hit_ratio` recover.
- On cloud APIs: translate each hit/miss into dollar amounts using the cost
  table from `llm_engines` README.

## Section 4 — RAG and the stable prefix

- Background RAG (content retrieved in nearly every query) should be pinned
  in the stable prefix zone, not re-retrieved dynamically each turn.
- Exercise: build a short RAG pipeline, identify which chunks recur across
  queries, pin them explicitly, and observe cache improvement.
- Contrast with per-query chunks (always dynamic, always after the stable zone).

## Section 5 — Context rot and the RLM pattern (advanced)

- Context rot: quality degradation as the window fills with dead ends and
  prior failed attempts in an agent loop.
- Two defences: Engram dedup (prune redundant entries) and the RLM pattern
  (keep large data external entirely).
- RLM concept: the model receives data as a Python variable and writes code
  to explore it selectively. Each LLM call stays small regardless of data
  volume. Reference: arXiv:2512.24601.
- Complexity classes where RLM matters most: linear (scan every item),
  quadratic (pairwise reasoning across items).
- Depth-1 constraint and the sandbox requirement (ADR-011 isolation baseline).
- Exercise: contrast direct-load vs RLM-style chunked exploration on a 50-page
  PDF. Measure token counts and output quality.

## Section 6 — Putting it together: the cost-aware application

A worked example combining all four levers:
1. Stable prefix (system prompt + pinned background RAG)
2. `session_id` on every request
3. Engram `TokenBudget` tuned to keep dynamic context within the cache boundary
4. Per-query RAG chunks in the dynamic zone
5. `CacheStats` monitored each turn as a cost/latency dashboard

Students compare a naively assembled prompt vs the optimised assembly on both
a local backend (latency) and a cloud API (cost estimate from `hit_ratio`).

---

## Air-gap note

All exercises run against Ollama with a local model. Sections 2 and 6 note
the cloud cost implications but do not require an API key. The cost tables
are computed from `cache_stats.hit_ratio` × an assumed price-per-token so
students can reason about economics without spending money.
